# Titanic
## Score: .77511

In [42]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from catboost import CatBoostClassifier
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

SEED = 42
np.random.seed(SEED)

_root = Path.cwd()
_data = _root / "titanic"

RARE = {
    "Lady",
    "Countess",
    "Capt",
    "Col",
    "Don",
    "Dr",
    "Major",
    "Rev",
    "Sir",
    "Jonkheer",
    "Dona",
}


def build_xy(train_df: pd.DataFrame, test_df: pd.DataFrame):
    y = train_df["Survived"].to_numpy()
    tr = train_df.drop(columns=["Survived"]).copy()
    te = test_df.copy()
    ticket_vc = tr["Ticket"].value_counts()
    med_fare = tr["Fare"].median()
    mode_emb = tr["Embarked"].mode().iloc[0]
    tr["Fare"] = tr["Fare"].fillna(med_fare)
    te["Fare"] = te["Fare"].fillna(med_fare)
    tr["Embarked"] = tr["Embarked"].fillna(mode_emb)
    te["Embarked"] = te["Embarked"].fillna(mode_emb)

    def impute_age(ref: pd.DataFrame, df: pd.DataFrame) -> None:
        gmed = ref["Age"].median()
        for (sex, pcl), m in ref.groupby(["Sex", "Pclass"])["Age"].median().items():
            mm = m if pd.notna(m) else gmed
            msk = (df["Sex"] == sex) & (df["Pclass"] == pcl) & df["Age"].isna()
            df.loc[msk, "Age"] = mm
        df["Age"] = df["Age"].fillna(gmed)

    impute_age(tr, tr)
    impute_age(tr, te)

    def feats(df: pd.DataFrame) -> pd.DataFrame:
        out = df.copy()
        out["Title"] = out["Name"].str.extract(r",\s*([^\.]+)\.", expand=False).str.strip()
        out["Title"] = out["Title"].replace({"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"})
        out["Title"] = out["Title"].where(~out["Title"].isin(RARE), "Rare")
        out["FamilySize"] = out["SibSp"] + out["Parch"] + 1
        out["IsAlone"] = (out["FamilySize"] == 1).astype(int)
        out["Deck"] = out["Cabin"].apply(
            lambda x: str(x)[0] if pd.notna(x) and str(x) and str(x)[0].isalpha() else "U"
        )
        out["TicketGroupSize"] = out["Ticket"].map(ticket_vc).fillna(1).astype(int)
        out["FarePerPerson"] = out["Fare"] / out["FamilySize"].clip(lower=1)
        out["HasCabin"] = out["Cabin"].notna().astype(int)
        out["AgeBin"] = pd.cut(
            out["Age"],
            bins=[0, 12, 18, 35, 60, 200],
            labels=["c", "t", "y", "a", "s"],
        ).astype(str)
        out = out.drop(columns=["Name", "Ticket", "Cabin", "PassengerId"], errors="ignore")
        out["Pclass"] = out["Pclass"].astype(str)
        for c in ["Sex", "Embarked", "Title", "Deck", "AgeBin"]:
            out[c] = out[c].astype(str)
        return out

    X = feats(tr)
    X_te = feats(te)
    return X, y, X_te


train_raw = pd.read_csv(_data / "train.csv")
test_raw = pd.read_csv(_data / "test.csv")
pid = test_raw["PassengerId"].values

X, y, X_test = build_xy(train_raw, test_raw)

cat_cols = ["Sex", "Embarked", "Title", "Deck", "Pclass", "AgeBin"]
num_cols = [c for c in X.columns if c not in cat_cols]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
n = len(X)
nt = len(X_test)
oof_cb = np.zeros(n)
oof_lgb = np.zeros(n)
oof_rf = np.zeros(n)
sum_cb = np.zeros(nt)
sum_lgb = np.zeros(nt)
sum_rf = np.zeros(nt)

for fold, (tr_idx, va_idx) in enumerate(cv.split(X, y)):
    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y[tr_idx], y[va_idx]

    cb = CatBoostClassifier(
        iterations=8000,
        learning_rate=0.025,
        depth=7,
        l2_leaf_reg=2.5,
        random_seed=SEED + fold,
        thread_count=1,
        verbose=False,
        loss_function="Logloss",
        eval_metric="AUC",
    )
    cb.fit(
        X_tr,
        y_tr,
        cat_features=cat_cols,
        eval_set=(X_va, y_va),
        early_stopping_rounds=150,
        verbose=False,
    )
    oof_cb[va_idx] = cb.predict_proba(X_va)[:, 1]
    sum_cb += cb.predict_proba(X_test)[:, 1]

    X_tr_l = X_tr.copy()
    X_va_l = X_va.copy()
    X_te_l = X_test.copy()
    for c in cat_cols:
        X_tr_l[c] = X_tr_l[c].astype("category")
        X_va_l[c] = X_va_l[c].astype("category")
        X_te_l[c] = X_te_l[c].astype("category")
    lgbm = lgb.LGBMClassifier(
        n_estimators=8000,
        learning_rate=0.025,
        num_leaves=48,
        max_depth=-1,
        subsample=0.88,
        colsample_bytree=0.85,
        reg_alpha=0.12,
        reg_lambda=1.2,
        random_state=SEED + fold,
        verbose=-1,
        n_jobs=1,
    )
    lgbm.fit(
        X_tr_l,
        y_tr,
        eval_set=[(X_va_l, y_va)],
        callbacks=[lgb.early_stopping(150, verbose=False), lgb.log_evaluation(0)],
    )
    oof_lgb[va_idx] = lgbm.predict_proba(X_va_l)[:, 1]
    sum_lgb += lgbm.predict_proba(X_te_l)[:, 1]

    pre_rf = ColumnTransformer(
        [
            ("n", SimpleImputer(strategy="median"), num_cols),
            (
                "c",
                Pipeline(
                    [
                        ("i", SimpleImputer(strategy="most_frequent")),
                        ("o", OneHotEncoder(handle_unknown="ignore", max_categories=30)),
                    ]
                ),
                cat_cols,
            ),
        ]
    )
    rf = Pipeline(
        [
            ("p", pre_rf),
            (
                "m",
                RandomForestClassifier(
                    n_estimators=600,
                    max_depth=14,
                    min_samples_leaf=2,
                    max_features="sqrt",
                    random_state=SEED + fold,
                    n_jobs=1,
                ),
            ),
        ]
    )
    rf.fit(X_tr, y_tr)
    oof_rf[va_idx] = rf.predict_proba(X_va)[:, 1]
    sum_rf += rf.predict_proba(X_test)[:, 1]

n_f = float(cv.n_splits)
oof_avg = (oof_cb + oof_lgb + oof_rf) / 3.0
test_avg = (sum_cb + sum_lgb + sum_rf) / (3.0 * n_f)

best_t, best_acc = 0.5, -1.0
for t in np.linspace(0.28, 0.72, 89):
    acc = accuracy_score(y, (oof_avg >= t).astype(int))
    if acc > best_acc:
        best_acc, best_t = acc, t

print(round(best_acc, 5), round(best_t, 4))

sub = (test_avg >= best_t).astype(int)
out = pd.DataFrame({"PassengerId": pid, "Survived": sub})
out.to_csv(_root / "submission.csv", index=False)
out.head(10)

0.844 0.55


,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,0
5,897,0
6,898,0
7,899,0
8,900,1
9,901,0
